# 节点 09 — 手撕 Transformer（注意力就是一切）

> 论文：Vaswani et al. (2017) *Attention Is All You Need* [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)

**本 notebook 的目标**：
1. 手撕 **缩放点积注意力**（Scaled Dot-Product Attention）— 只用 numpy
2. 手撕 **多头注意力**（Multi-Head Attention）— 只用 numpy
3. 可视化 **位置编码**（Positional Encoding）— 理解时钟叠加原理
4. 端到端演示：用 mini-Transformer 编码器处理一条小句子

**前置知识**：节点07（Bahdanau Attention），会矩阵乘法，懂 softmax。

**不需要**：PyTorch、TensorFlow——只用 Python + NumPy。

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')  # 无界面后端，允许 nbconvert --execute 运行
import matplotlib.pyplot as plt

np.random.seed(42)  # 固定随机种子，结果可重复
print('NumPy 版本:', np.__version__)
print('准备好了！')

---

## 第一部分：缩放点积注意力

### 从 Bahdanau 到 Transformer：换一种打分方式

节点07里，Bahdanau 注意力用**加法打分**：

$$e_j = \mathbf{v}^\top \tanh(W_1 h_j + W_2 s)$$

Transformer 换成了**点积打分**，并做了缩放：

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

### 三个角色：Q、K、V（图书馆类比）

想象你在一个图书馆：
- **Q（Query，查询）**：你手里的搜索关键词
- **K（Key，索引）**：每本书封面上的标题/标签
- **V（Value，内容）**：每本书的实际内容

过程：
1. 用查询词 Q 对比每本书的标签 K → 得到相关度分数
2. 用 softmax 把分数变成权重（所有书的权重加起来 = 1）
3. 按权重提取内容 V 的加权平均 → 得到你想要的答案

### 为什么要除以 $\sqrt{d_k}$？

点积 $Q \cdot K$ 的数量级随维度增大而增大。
如果 $d_k = 64$，点积结果的标准差约为 $\sqrt{64} = 8$。

不缩放的话，分数会很大 → softmax 梯度接近 0 → 训练困难。
除以 $\sqrt{d_k}$ 把方差拉回到 1，softmax 工作正常。

In [ ]:
def softmax(x, axis=-1):
    """数值稳定的 softmax（减去最大值防溢出）。"""
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    缩放点积注意力（Scaled Dot-Product Attention）。

    参数：
        Q: shape (..., seq_q, d_k)  查询矩阵
        K: shape (..., seq_k, d_k)  键矩阵
        V: shape (..., seq_k, d_v)  值矩阵
        mask: shape (..., seq_q, seq_k)  True 的位置被遮住

    返回：
        output:  shape (..., seq_q, d_v)
        weights: shape (..., seq_q, seq_k)
    """
    d_k = Q.shape[-1]

    # 步骤1：点积打分，除以 sqrt(d_k) 缩放
    scores = Q @ K.swapaxes(-2, -1) / np.sqrt(d_k)

    # 步骤2：遮掩（用于解码器的遮掩自注意力）
    if mask is not None:
        scores = np.where(mask, -1e9, scores)

    # 步骤3：softmax → 注意力权重
    weights = softmax(scores, axis=-1)

    # 步骤4：加权求和
    output = weights @ V

    return output, weights


print('scaled_dot_product_attention 定义完成')

### 验证：单头注意力的形状和数学性质

In [ ]:
seq_len = 4
d_k = 8
d_v = 8

Q_test = np.random.randn(seq_len, d_k)
K_test = np.random.randn(seq_len, d_k)
V_test = np.random.randn(seq_len, d_v)

output, weights = scaled_dot_product_attention(Q_test, K_test, V_test)

print('=== 形状验证 ===')
print(f'输出形状: {output.shape}    （期望: ({seq_len}, {d_v})）')
print(f'权重形状: {weights.shape}  （期望: ({seq_len}, {seq_len})）')

row_sums = weights.sum(axis=-1)
assert np.allclose(row_sums, 1.0, atol=1e-7), f'权重行和不为1: {row_sums}'
print('断言通过：每行权重之和 = 1')

assert (weights >= 0).all()
print('断言通过：所有注意力权重 >= 0')

assert output.shape == (seq_len, d_v)
assert np.isfinite(output).all()
print('断言通过：输出形状正确，无 NaN/Inf')

### 验证：缩放的作用

不缩放时，softmax 会出现 winner-take-all（一个位置拿走几乎全部权重）。

In [ ]:
d_k_large = 64
q = np.random.randn(d_k_large)
k = np.random.randn(d_k_large)

dot_product = q @ k
scaled_dot  = dot_product / np.sqrt(d_k_large)

print(f'd_k = {d_k_large}')
print(f'原始点积: {dot_product:.2f}')
print(f'缩放后点积: {scaled_dot:.2f}')
print()

dots_raw    = np.array([dot_product, 0.0, -dot_product/2, dot_product/3])
dots_scaled = dots_raw / np.sqrt(d_k_large)

w_raw    = softmax(dots_raw)
w_scaled = softmax(dots_scaled)

print('未缩放 softmax（容易出现 winner-take-all）:')
print('  ', w_raw.round(4))
print(f'  最大-次大差值: {w_raw.max() - np.sort(w_raw)[-2]:.4f}')

print('缩放后 softmax（分布更均匀，梯度更健康）:')
print('  ', w_scaled.round(4))
print(f'  最大-次大差值: {w_scaled.max() - np.sort(w_scaled)[-2]:.4f}')

---

## 第二部分：多头注意力

### 为什么需要多个「头」？

单头注意力一次只能关注一种关系（比如句法关系）。
多头注意力同时学习多种关系：
- 第1头：关注相邻词的语法关系
- 第2头：关注同指（代词 → 名词）
- 第3头：关注动词 → 宾语
- ……每个头自己学，各有侧重

### 公式

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$

其中每个头独立计算：

$$\text{head}_i = \text{Attention}(Q W_i^Q, K W_i^K, V W_i^V)$$

原始 Transformer：$d_{\text{model}} = 512$，$h = 8$，$d_k = d_v = 64$。

In [ ]:
class MultiHeadAttention:
    """
    多头注意力（纯 NumPy 实现）。
    d_model 必须整除 num_heads。
    """
    def __init__(self, d_model, num_heads):
        assert d_model % num_heads == 0, (
            f'd_model({d_model}) 必须能被 num_heads({num_heads}) 整除')
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        scale = np.sqrt(2.0 / (d_model + self.d_k))
        self.WQ = np.random.randn(d_model, d_model) * scale
        self.WK = np.random.randn(d_model, d_model) * scale
        self.WV = np.random.randn(d_model, d_model) * scale
        self.WO = np.random.randn(d_model, d_model) * scale

    def split_heads(self, x):
        """(seq_len, d_model) → (num_heads, seq_len, d_k)"""
        seq_len = x.shape[0]
        x = x.reshape(seq_len, self.num_heads, self.d_k)
        return x.transpose(1, 0, 2)

    def forward(self, Q_in, K_in, V_in, mask=None):
        """
        Q_in, K_in, V_in: shape (seq, d_model)
        返回: output (seq_q, d_model), all_weights (num_heads, seq_q, seq_k)
        """
        Q_proj = Q_in @ self.WQ
        K_proj = K_in @ self.WK
        V_proj = V_in @ self.WV

        Q_heads = self.split_heads(Q_proj)
        K_heads = self.split_heads(K_proj)
        V_heads = self.split_heads(V_proj)

        mask_h = mask[np.newaxis, :, :] if mask is not None else None

        attn_output, all_weights = scaled_dot_product_attention(
            Q_heads, K_heads, V_heads, mask=mask_h
        )

        seq_q = Q_in.shape[0]
        concat = attn_output.transpose(1, 0, 2).reshape(seq_q, self.d_model)
        output = concat @ self.WO
        return output, all_weights


print('MultiHeadAttention 定义完成')

### 验证多头注意力的形状和数学性质

In [ ]:
d_model = 16
num_heads = 4
seq_len = 5

mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
X = np.random.randn(seq_len, d_model)

output, all_weights = mha.forward(X, X, X)  # 自注意力：Q=K=V=X

print('=== 多头注意力形状验证 ===')
print(f'输入 X 形状: {X.shape}')
print(f'输出形状: {output.shape}    （期望: ({seq_len}, {d_model})）')
print(f'权重形状: {all_weights.shape}  （期望: ({num_heads}, {seq_len}, {seq_len})）')

weight_row_sums = all_weights.sum(axis=-1)
assert np.allclose(weight_row_sums, 1.0, atol=1e-6)
print(f'断言通过：{num_heads} 头 × {seq_len} 行，所有权重行和 = 1')

assert output.shape == (seq_len, d_model)
assert np.isfinite(output).all()
print('断言通过：输出形状正确，无 NaN/Inf')

### 可视化：各个头的注意力矩阵热力图

In [ ]:
words = ['The', 'quick', 'brown', 'fox', 'jumps']
seq_len_vis = len(words)
d_model_vis = 16
num_heads_vis = 4

mha_vis = MultiHeadAttention(d_model=d_model_vis, num_heads=num_heads_vis)
X_vis = np.random.randn(seq_len_vis, d_model_vis)
_, weights_vis = mha_vis.forward(X_vis, X_vis, X_vis)

fig, axes = plt.subplots(1, num_heads_vis, figsize=(3 * num_heads_vis, 3.2))
for i, ax in enumerate(axes):
    ax.imshow(weights_vis[i], cmap='Blues', vmin=0, vmax=weights_vis[i].max())
    ax.set_title(f'头 {i+1}', fontsize=10)
    ax.set_xticks(range(seq_len_vis))
    ax.set_yticks(range(seq_len_vis))
    ax.set_xticklabels(words, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(words, fontsize=8)
    for r in range(seq_len_vis):
        for c in range(seq_len_vis):
            ax.text(c, r, f'{weights_vis[i, r, c]:.2f}',
                    ha='center', va='center', fontsize=7,
                    color='white' if weights_vis[i, r, c] > 0.35 else 'black')

plt.suptitle('多头自注意力权重（随机初始化，训练后每头会专注不同语言关系）',
             fontsize=10, y=1.02)
plt.tight_layout()
plt.savefig('mha_weights.png', dpi=80, bbox_inches='tight')
plt.close()
print('多头注意力权重热力图已保存 -> mha_weights.png')

---

## 第三部分：位置编码

### 为什么需要位置编码？

自注意力计算时，**每个词同时看所有词**——没有顺序信息。
「猫追狗」和「狗追猫」的词嵌入一样，但意思完全相反！

解决方法：给每个位置加上一个**独特的向量**，让模型知道每个词的位置（第 $i$ 个词）。

### 时钟叠加原理

$$\text{PE}(pos, 2i)   = \sin\!\left(\frac{pos}{10000^{2i/d}}\right)$$
$$\text{PE}(pos, 2i+1) = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

直觉：不同位置有不同频率的时钟：
- 低维度（$i$ 小）：高频，区分相邻位置（像秒针）
- 高维度（$i$ 大）：低频，区分远距位置（像小时针）

In [ ]:
def positional_encoding(max_seq_len, d_model):
    """生成位置编码矩阵 shape (max_seq_len, d_model)。"""
    PE = np.zeros((max_seq_len, d_model))
    positions = np.arange(max_seq_len)[:, np.newaxis]   # (max_seq_len, 1)
    i = np.arange(0, d_model, 2)[np.newaxis, :]          # (1, d_model//2)
    freqs = 1.0 / (10000 ** (i / d_model))
    angles = positions * freqs                           # (max_seq_len, d_model//2)
    PE[:, 0::2] = np.sin(angles)
    PE[:, 1::2] = np.cos(angles)
    return PE


max_seq_len = 50
d_model_pe = 32
PE = positional_encoding(max_seq_len, d_model_pe)

print(f'位置编码形状: {PE.shape}')
print(f'值域: [{PE.min():.3f}, {PE.max():.3f}]  （应在 [-1,1] 内）')

assert PE.min() >= -1.0 - 1e-9 and PE.max() <= 1.0 + 1e-9
print('断言通过：值在 [-1, 1] 内（sin/cos 的值域）')

assert not np.allclose(PE[0], PE[1])
assert not np.allclose(PE[0], PE[10])
print('断言通过：不同位置的编码向量不相同')

### 可视化位置编码热力图和波形

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
im = ax.imshow(PE[:20, :].T, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
ax.set_xlabel('位置（序列中第几个词）', fontsize=11)
ax.set_ylabel('编码维度', fontsize=11)
ax.set_title('位置编码热力图（前20个位置）', fontsize=11)
plt.colorbar(im, ax=ax, label='值')

ax2 = axes[1]
colors = plt.cm.viridis(np.linspace(0, 1, 4))
for dim_idx, color in zip([0, 4, 8, 16], colors):
    ax2.plot(np.arange(50), PE[:, dim_idx],
             label=f'维度 {dim_idx}', color=color, linewidth=1.5)

ax2.set_xlabel('位置', fontsize=11)
ax2.set_ylabel('sin/cos 值', fontsize=11)
ax2.set_title('不同维度的波形\n（低维高频，高维低频）', fontsize=11)
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('positional_encoding.png', dpi=80, bbox_inches='tight')
plt.close()
print('位置编码可视化已保存 -> positional_encoding.png')

### 验证：位置编码能让模型感知相对距离

相邻位置的 PE 向量余弦相似度，应比远距位置更高。

In [ ]:
pe_large = positional_encoding(50, 512)  # 用论文维度 512
pe_0 = pe_large[0]

cosine_sims = np.array([
    (pe_large[i] @ pe_0) / (np.linalg.norm(pe_large[i]) * np.linalg.norm(pe_0))
    for i in range(50)
])

print('位置 0 与各位置的余弦相似度（d_model=512）:')
for pos in [0, 1, 5, 20, 49]:
    print(f'  位置 {pos:2d}: {cosine_sims[pos]:.4f}')

assert abs(cosine_sims[0] - 1.0) < 1e-8
print('断言通过：自相似度 = 1')

sim_near = cosine_sims[1:6].mean()
sim_far  = cosine_sims[20:30].mean()
assert sim_near > sim_far, (
    f'近邻相似度({sim_near:.4f}) 应 > 远距相似度({sim_far:.4f})')
print(f'断言通过：近邻({sim_near:.4f}) > 远距({sim_far:.4f})')
print('位置编码让模型能感知相对距离')

---

## 第四部分：一层完整的 Transformer 编码器

每一层编码器包含两个子层：
1. **多头自注意力** + 残差连接 + Layer Norm
2. **前馈网络（FFN）**先扩展（512→2048），ReLU，再压缩（2048→512）
   + 残差连接 + Layer Norm

$$x_1 = \text{LayerNorm}(x + \text{MultiHeadAttn}(x, x, x))$$
$$x_2 = \text{LayerNorm}(x_1 + \text{FFN}(x_1))$$

**残差连接**（输出 = 输入 + 变换）防止梯度消失，让深层网络也能训练。

In [ ]:
def layer_norm(x, eps=1e-6):
    """Layer Norm：每个位置独立归一化，均值→0，方差→1。"""
    mean = x.mean(axis=-1, keepdims=True)
    var  = x.var(axis=-1, keepdims=True)
    return (x - mean) / np.sqrt(var + eps)


def ffn(x, d_model, d_ff=None):
    """前馈网络：先扩展(×4) ReLU 再压缩，增加非线性变换能力。"""
    if d_ff is None:
        d_ff = d_model * 4  # 论文：4 倍扩展（512 → 2048）
    W1 = np.random.randn(d_model, d_ff)  * np.sqrt(2.0 / (d_model + d_ff))
    W2 = np.random.randn(d_ff,   d_model) * np.sqrt(2.0 / (d_ff + d_model))
    h = np.maximum(0, x @ W1)  # 扩展 + ReLU
    return h @ W2              # 压缩


class TransformerEncoderLayer:
    """一层 Transformer 编码器：MHA + FFN，两个子层都有残差 + LayerNorm。"""
    def __init__(self, d_model, num_heads):
        self.d_model = d_model
        self.mha = MultiHeadAttention(d_model, num_heads)

    def forward(self, x):
        """x: (seq_len, d_model) → (seq_len, d_model)"""
        # 子层1：多头自注意力 + 残差 + LayerNorm
        attn_out, _ = self.mha.forward(x, x, x)
        x = layer_norm(x + attn_out)

        # 子层2：FFN + 残差 + LayerNorm
        x = layer_norm(x + ffn(x, self.d_model))
        return x


print('layer_norm / ffn / TransformerEncoderLayer 定义完成')

### 端到端演示：用 mini-Transformer 编码一句话

In [ ]:
d_model = 16
num_heads = 4
n_layers = 2

sentence = ['The', 'quick', 'brown', 'fox', 'jumps']
seq_len = len(sentence)

# 词嵌入（随机初始化，模拟）+ 位置编码
embeddings = np.random.randn(seq_len, d_model)
PE_matrix  = positional_encoding(seq_len, d_model)
x = embeddings + PE_matrix

print(f'输入句子: {sentence}')
print(f'词嵌入 + 位置编码形状: {x.shape}')
print()

for layer_i in range(n_layers):
    enc = TransformerEncoderLayer(d_model=d_model, num_heads=num_heads)
    x = enc.forward(x)

    per_pos_mean = x.mean(axis=-1)
    per_pos_std  = x.std(axis=-1)
    assert np.allclose(per_pos_mean, 0.0, atol=1e-5), (
        f'层{layer_i+1} 均值不为0: {per_pos_mean}')
    assert np.allclose(per_pos_std, 1.0, atol=1e-3), (
        f'层{layer_i+1} 标准差不为1: {per_pos_std}')
    print(f'第{layer_i+1}层: 形状={x.shape}, 均值={x.mean():.4f}, std={x.std():.4f}')

print()
print(f'最终编码器输出: {x.shape}')
print('断言通过：每层 LayerNorm 后均值 ≈ 0，标准差 ≈ 1')

---

## 第五部分：因果遮掩（解码器专用）

解码器生成第 $t$ 个词时，只能看第 1 到第 $t-1$ 个词，不能「偷看」未来。

实现：构造上三角 mask，被遮掩位置分数设为 $-10^9$（softmax 后接近 0）。

In [ ]:
def causal_mask(seq_len):
    """上三角因果遮掩：mask[i,j]=True 表示位置 i 不能看 j（j>i 时）。"""
    return np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)


mask_demo = causal_mask(4)
print('因果遮掩矩阵（True = 被遮掩）:')
print(mask_demo)
print()
print('位置0（第1词）: 只能看自己')
print('位置1（第2词）: 能看第1、2词')
print('位置3（第4词）: 能看全部4词')

lower = np.tril(np.ones((4, 4), dtype=bool), k=0)
upper = np.triu(np.ones((4, 4), dtype=bool), k=1)
assert not mask_demo[lower].any()
assert mask_demo[upper].all()
print('断言通过：下三角未被遮掩，上三角全部遮掩')

Q_m = np.random.randn(4, 8)
K_m = np.random.randn(4, 8)
V_m = np.random.randn(4, 8)
_, w_masked = scaled_dot_product_attention(Q_m, K_m, V_m, mask=causal_mask(4))

print('遮掩后注意力权重（上三角应接近0）:')
print(w_masked.round(6))

for i in range(4):
    for j in range(i + 1, 4):
        assert w_masked[i, j] < 1e-6, (
            f'位置({i},{j}) 应被遮掩，权重={w_masked[i,j]}')
print('断言通过：所有未来位置权重 < 1e-6，解码器无法偷看未来词')

---

## 总结

| 组件 | 实现 | 验证 |
|------|------|------|
| 缩放点积注意力 | `scaled_dot_product_attention` | 形状、权重行和=1、无NaN |
| 多头注意力 | `MultiHeadAttention` | 多头形状、权重行和=1 |
| 位置编码 | `positional_encoding` | 值域[-1,1]、不同位置不同、近邻更相似 |
| Layer Norm | `layer_norm` | 均值≈0、标准差≈1 |
| 前馈网络 | `ffn` | 先扩展×4再压缩 |
| 因果遮掩 | `causal_mask` | 未来位置权重<1e-6 |
| 编码器层 | `TransformerEncoderLayer` | 端到端形状正确、LayerNorm有效 |

### 与节点07（Bahdanau Attention）的对比

| 维度 | Bahdanau（节点07） | Transformer（本节） |
|------|------------------|--------------------|
| 打分 | 加性 tanh 函数 | 点积除以 sqrt(d_k) |
| 并行 | 顺序（依赖 RNN） | **完全并行** |
| 多头 | 单注意力 | **多头，多种关系** |
| 位置 | RNN 天然携带 | **显式位置编码** |

**下一站**：节点 10 — BERT，用 Transformer 编码器做大规模自监督预训练。